In [1]:
import csv
import pandas as pd #imports pandas, library for working with tables

## Cleaning the TP53 mutation database files

In [2]:
#load the data
p53_df = pd.read_csv('TP53 mutations.csv')

#count mutations per cell line
mutations_per_cell_line = p53_df['Lineage'].value_counts()

#group by mutation type and count
mutations_by_type = p53_df.groupby('Variant Info').size()

#group by mutation type and cell line, then count
mutations_by_type_and_line = p53_df.groupby(['Variant Info', 'Lineage']).size()

#identify the most common mutation type
most_common_mutation = mutations_by_type.idxmax()

#mutation of interest - change this to the mutation you want to analyze!!
mutation_of_interest = 'p.R175H'

#filter rows with the mutation of interest
mutation_df = p53_df[p53_df['Protein Change'].str.strip() == mutation_of_interest]

#get unique cell line names and their DepMap IDs
mutation_results = mutation_df[['Cell Line', 'Lineage']].drop_duplicates()

mutation_results['Lineage'] = (
    mutation_results['Lineage']
    .str.replace('Fallopian Tube', '', regex=False)
    .str.replace('Bladder', '', regex=False)
    .str.replace('Esophagus', '', regex=False)
    .str.replace(' ', '_', regex=False)   #join multi-word lineages
    .str.replace(r'\s{2,}', ' ', regex=True)   #collapse extra spaces
    .str.strip(' /-')                          #trim separators left behind
    .str.upper()                               #capitalize
)

mutation_results_output = f'p53_{mutation_of_interest}_results.tsv'
mutation_results.to_csv(mutation_results_output, sep='\t', index=False, na_rep='nan')

## Cleaning the TP53 wildtype database files

In [ ]:
#load the data
tp53_wt = pd.read_csv('tp53db_cell_lines_r21.csv')

#remove cell lines that do not have a DepMap ID
tp53_wt = tp53_wt[tp53_wt['depmap_ID'].notna()]

#make three dictionaries in one pass through Model.csv
# - DepMap ID -> CCLE name
# - CCLE name -> DepMap ID
# - DepMap ID -> Stripped Cell Line name
DepmapID_to_CCLE_name = {}
CCLE_name_to_DepmapID = {}
DepMapID_to_CellLineName = {}

with open('Model.csv', 'r') as file:
    csvFile = csv.reader(file)
    next(csvFile, None)  # skip header
    for line in csvFile:
        if len(line) < 5:
            continue

        ccle_name = line[40].strip()
        cell_line_name = line[3].strip()
        depmap_id = line[0].strip()

        if not ccle_name or not depmap_id:
            continue

        CCLE_name_to_DepmapID.setdefault(ccle_name, depmap_id)
        DepmapID_to_CCLE_name.setdefault(depmap_id, ccle_name)
        if cell_line_name:
            DepMapID_to_CellLineName.setdefault(depmap_id, cell_line_name)

tp53_wt = tp53_wt.assign(**{'CCLE name': tp53_wt['depmap_ID'].map(DepmapID_to_CCLE_name)})
tp53_wt = tp53_wt.assign(**{'Cell Line Name': tp53_wt['depmap_ID'].map(DepMapID_to_CellLineName)})
tp53_wildtype = pd.DataFrame(tp53_wt[['CCLE name', 'Cell Line Name', 'depmap_ID']])

tp53_wildtype_output = tp53_wildtype.to_csv('tp53_wildtype_cell_lines.tsv', index=False, sep='\t')

## Import necessary files

In [4]:
gene_exp_matrix = pd.read_csv('ordered_gene_expression_matrix_PRISM532.tsv', sep='\t', index_col=0)
drug_response_matrix = pd.read_csv('ordered_drug_matrix_PRISM532.tsv', sep='\t', index_col=0)
splicing_matrix = pd.read_csv('ordered_juncbase_matrix_PRISM532.tsv', sep='\t', index_col=0)

## Mutant Pre-processing

In [5]:
mut_list = []
mutation_of_interest = 'R175H' #change this to the mutation you want to analyze!!
with open(f'p53_p.{mutation_of_interest}_results.tsv', 'r') as f:
    for line in f:
        line_items = line.strip().split('\t')
        if line_items[1] == 'Lineage':
            cell_line_name = line_items[0] + '_' + line_items[1]
        else:
            cell_line_name = line_items[0] + '_' + line_items[1].upper()
        mut_list.append(cell_line_name)

#gene expression 
cell_lines = [x for x in mut_list if x in gene_exp_matrix.columns]
tp53_mut_gene_exp = gene_exp_matrix[cell_lines]

tp53_mut_gene_exp.to_csv(f'{mutation_of_interest}_gene_expression.tsv', sep='\t')

#drug response
id_to_name = {}
with open('ordered_drug_matrix_PRISM532.tsv', 'r') as f:
    for line in f:
        line_items = line.strip().split('\t')
        id_to_name[line_items[0]] = line_items[1]
    id_to_name.pop('drug_name', None) #remove header
cell_lines = tp53_mut_gene_exp.columns
tp53_mut_drug_res = drug_response_matrix.loc[:, cell_lines]
tp53_mut_drug_res.rename(index=id_to_name, inplace=True)

tp53_mut_drug_res.to_csv(f'{mutation_of_interest}_drug_response.tsv', sep='\t')

#splicing
cell_lines = tp53_mut_gene_exp.columns 
tp53_mut_splicing = splicing_matrix.loc[:, cell_lines]

tp53_mut_splicing.to_csv(f'{mutation_of_interest}_splicing.tsv', sep='\t')

## Wildtype Pre-processing

In [11]:
wt_list = []
with open(f'tp53_wildtype_cell_lines.tsv', 'r') as f:
    for line in f:
        line_items = line.strip().split('\t')
        wt_list.append(line_items[0])
    wt_list = wt_list[1:] #remove header

#gene expression
cell_lines = [x for x in wt_list if x in gene_exp_matrix.columns]
tp53_wt_gene_exp = gene_exp_matrix.loc[:, cell_lines]

tp53_wt_gene_exp.to_csv(f'wildtype_gene_expression.tsv', sep='\t')

#drug response
tp53_wt_drug_res = drug_response_matrix.loc[:, tp53_wt_gene_exp.columns]
tp53_wt_drug_res.rename(index=id_to_name, inplace=True)

tp53_wt_drug_res.to_csv(f'wildtype_drug_response.tsv', sep='\t')


#splicing
tp_wt_splicing = splicing_matrix.loc[:, tp53_wt_gene_exp.columns]

tp_wt_splicing.to_csv(f'wildtype_splicing.tsv', sep='\t')

## Wilcoxon Rank-Sum

In [ ]:
"""
PROGRAM OVERVIEW:
-----------------
1. Load mutant and WT data
2. Match shared features
3. For each feature:
    - collect mutant values
    - collect WT values
    - run Wilcoxon rank-sum test
    - compute median difference
4. Correct all p-values for multiple testing
5. Save the results to CSV
6. Repeat for:
    - drug response
    - gene expression
    - splicing
    - across three TP53 mutations

feature: drug, gene, or splicing event being tested
delta_value: the median difference
p_value: raw Wilcoxon rank-sum p-value
corrected_p_value: Benjamini-Hochberg adjusted p-value

"""

from scipy.stats import ranksums # imports  Wilcoxon rank-sum test from SciPy
from statsmodels.stats.multitest import multipletests # imports function that corrects p-values for multiple testing

# create function that computes rank some and that can be used for all 3 files
def run_rank_sum(mut_df, wt_df, output_name): # inputs: mutant data table, wildtype data table, output file name
    """
    Compare mutant vs WT for every feature row in a dataframe.
    Saves a CSV with:
    name of what you're testing, delta_value (median(mutant) − median(wildtype)), p_value, corrected_p_value
    """

    common_features = mut_df.index.intersection(wt_df.index) # finds all row names that appear in both tables
    # row names are drug names, gene names, feature ids
    # needed because can only compare a feature if it exists in both mutant and WT

    # keeps only shared rows in each table, now mutant and WT have the same set of features
    mut_df = mut_df.loc[common_features]
    wt_df = wt_df.loc[common_features]

    results = [] # create empty list for final results table

    for feature in common_features: # loops through one feature at a time
        
        # grab row for that feature from both tables
        # double brackets force pandas to return a DataFrame (2D table) not a Series (1D array)
        mut_row = mut_df.loc[[feature]]
        wt_row = wt_df.loc[[feature]]

        # use the first row and convert to numeric arrays
        # mut_row.iloc[0] takes first row of one-row DataFrame and turns it into a Series
        # pd.to_numeric(..., errors="coerce") converts values to numbers
        # .dropna() removes missing values
        # .values turns panda Series into a NumPy array
        
        mut_values = pd.to_numeric(mut_row.iloc[0], errors="coerce").dropna().values
        wt_values = pd.to_numeric(wt_row.iloc[0], errors="coerce").dropna().values

        # now, mut_values and wt_values are plain numeric lists which is needed for ranksums()

        # skips rows with too little data (statistical comparison with almost no data is unreliable)
        if len(mut_values) < 2 or len(wt_values) < 2: # if either group has less than 2 values, skip that feature
            continue # skip this feature and move to the next one

        # RUN WILCOXON RANK SUM TEST
        # compares mutant values to WT values for that one feature
        # smaller p-value = stronger evidence that mutant and WT differ
        
        stat, p_value = ranksums(mut_values, wt_values) # returns stat (test statistic) and p-value
        delta = pd.Series(mut_values).median() - pd.Series(wt_values).median() # computes the median difference
        # positive delta --> mutant values are generally higher
        # negative delta --> mutant values are generally lower

        results.append([feature, delta, p_value]) # adds current feature’s result to the list

    # turns list into a DataFrame
    # now have table with columns feature, delta value, p-value
    results_df = pd.DataFrame(results, columns=["feature", "delta_value", "p_value"])

    # correcting the p-value
    # adds a new column called corrected_p_value
    # results_df["p_value"] selects the column of raw p-values
    # multipletests(..., method="fdr_bh") applies Benjamini-Hochberg false discovery rate correction
    # [1] takes the second item from the function output to get adjusted p-values
    
    results_df["corrected_p_value"] = multipletests(results_df["p_value"], method="fdr_bh")[1]

    # sorting results
    # sorts results by smallest corrected p-value first, then by delta value
    # most interesting rows show up near the top
    results_df = results_df.sort_values(by=["corrected_p_value", "delta_value"])

    # writes the full table to a CSV file (contains all results)
    # index=False means don’t save the row numbers as an extra column
    results_df.to_csv(output_name, index=False)

    # prints a preview including output file name and the first 10 rows of the sorted results table
    print("\nResults for", output_name)
    print(results_df.head(10))


# ---------------- DRUG RESPONSE ----------------

r175h_drug = pd.read_csv("drug_response/R175H_drug_response.tsv", sep="\t", index_col=0) # loads the R175H drug-response file
r248q_drug = pd.read_csv("drug_response/R248Q_drug_response.tsv", sep="\t", index_col=0) # loads the R248Q drug-response file
r273h_drug = pd.read_csv("drug_response/R273H_drug_response.tsv", sep="\t", index_col=0) # loads the R273H drug-response file
wt_drug = pd.read_csv("drug_response/wildtype_drug_response.tsv", sep="\t", index_col=0) # loads the wildtype drug-response file
# sep="\t" means the file is tab-separated
# index_col=0 means the first column becomes the row labels

run_rank_sum(r175h_drug, wt_drug, "R175H_vs_WT_drug_results.csv") # runs rank-sum pipeline on R175H drug data vs WT drug data, saves the results
run_rank_sum(r248q_drug, wt_drug, "R248Q_vs_WT_drug_results.csv") # runs rank-sum pipeline on R248Q drug data vs WT drug data, saves the results
run_rank_sum(r273h_drug, wt_drug, "R273H_vs_WT_drug_results.csv") # runs rank-sum pipeline on R273H drug data vs WT drug data, saves the results


# ---------------- GENE EXPRESSION ----------------

# load the gene-expression files (same logic as before)
r175h_gene = pd.read_csv("gene_expression/R175H_gene_expression.tsv", sep="\t", index_col=0)
r248q_gene = pd.read_csv("gene_expression/R248Q_gene_expression.tsv", sep="\t", index_col=0)
r273h_gene = pd.read_csv("gene_expression/R273H_gene_expression.tsv", sep="\t", index_col=0)
wt_gene = pd.read_csv("gene_expression/wildtype_gene_expression.tsv", sep="\t", index_col=0)

# call rank-sum three times again
run_rank_sum(r175h_gene, wt_gene, "R175H_vs_WT_gene_results.csv")
run_rank_sum(r248q_gene, wt_gene, "R248Q_vs_WT_gene_results.csv")
run_rank_sum(r273h_gene, wt_gene, "R273H_vs_WT_gene_results.csv")


# ---------------- SPLICING ----------------
# splicing files have repeated gene names in first column, so don't use index_col=0 and run file normally
# instead, create a unique feature_id for each row.

r175h_splice = pd.read_csv("splicing/R175H_splicing.tsv", sep="\t")
r248q_splice = pd.read_csv("splicing/R248Q_splicing.tsv", sep="\t")
r273h_splice = pd.read_csv("splicing/R273H_splicing.tsv", sep="\t")
wt_splice = pd.read_csv("splicing/wildtype_splicing.tsv", sep="\t")

# create unique row IDs
for df in [r175h_splice, r248q_splice, r273h_splice, wt_splice]: # loops through all four splicing DataFrames
    df["feature_id"] = df["gene_name"] + "_" + df.index.astype(str) # creates a unique ID for each row by combining the gene name and row number
    df.set_index("feature_id", inplace=True) # makes feature_id the row index, inplace=True means change the DataFrame directly
    df.drop(columns=["gene_name"], inplace=True) # removes  original gene_name column b/c now the unique ID is being used instead

# call rank-sum three times again
run_rank_sum(r175h_splice, wt_splice, "R175H_vs_WT_splicing_results.csv")
run_rank_sum(r248q_splice, wt_splice, "R248Q_vs_WT_splicing_results.csv")
run_rank_sum(r273h_splice, wt_splice, "R273H_vs_WT_splicing_results.csv")

## Volcano Plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from adjustText import adjust_text

''' Mutation: R175H'''

# ---------------- DRUG RESPONSE ----------------

# Load the drug response data
df = pd.read_csv("R175H_vs_WT_drug_results.csv")

# Calculate -log10(p-value)
df["neg_log10_p"] = -np.log10(df["p_value"])

# Set thresholds
p_threshold = 0.05
effect_threshold = 1

# Classify points
conditions = [
    (df["p_value"] < p_threshold) & (df["delta_value"] > effect_threshold),
    (df["p_value"] < p_threshold) & (df["delta_value"] < -effect_threshold)
]
choices = ["More Sensitive", "More Resistant"]
df["significance"] = np.select(conditions, choices, default="Not Significant")

# ── Figure & axes ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ── Scatter points ─────────────────────────────────────────────────────────────
color_map = {
    "More Sensitive": "#C0392B",
    "More Resistant": "#2980B9",
    "Not Significant": "#AAAAAA",
}

order = ["Not Significant", "More Resistant", "More Sensitive"]
for category in order:
    subset = df[df["significance"] == category]
    ax.scatter(
        subset["delta_value"],
        subset["neg_log10_p"],
        c=color_map[category],
        s=18,
        alpha=0.75,
        linewidths=0,
        zorder=3 if category != "Not Significant" else 2,
    )

# ── Threshold lines ────────────────────────────────────────────────────────────
log10_p = -np.log10(p_threshold)

ax.axhline(log10_p, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(-effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(0, color="#888888", linestyle="--", linewidth=0.9, zorder=1)

# ── Drug name labels ───────────────────────────────────────────────────────────
labeled = df[df["significance"] != "Not Significant"]

texts = []
for _, row in labeled.iterrows():
    texts.append(ax.text(
        row["delta_value"],
        row["neg_log10_p"],
        row["feature"],
        fontsize=7,
        color=color_map[row["significance"]],
        va="bottom",
    ))

adjust_text(
    texts,
    x=df["delta_value"].values,
    y=df["neg_log10_p"].values,
    arrowprops=dict(
        arrowstyle="-",
        color="gray",
        lw=0.5,
        shrinkA=5,
        shrinkB=5,
    ),
    expand_points=(1.5, 1.5),
    expand_text=(1.2, 1.2),
    force_points=(0.5, 0.5),
    force_text=(0.5, 0.5),
)

# ── "Threshold" label ──────────────────────────────────────────────────────────
ax.text(
    ax.get_xlim()[1] if ax.get_xlim()[1] > effect_threshold + 0.5 else effect_threshold + 0.5,
    log10_p + 0.05,
    "Threshold",
    color="#888888", fontsize=8, va="bottom", ha="right",
)

# ── Zero-point arrows ──────────────────────────────────────────────────────────
ax.annotate(
    "", xy=(0.55, 0.97), xytext=(0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.annotate(
    "", xy=(-0.55, 0.97), xytext=(-0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.text(0, 0.975, "Zero point", ha="center", va="bottom",
        transform=ax.get_xaxis_transform(), fontsize=8, color="gray")

# ── Top annotation labels (moved below threshold line, outside data area) ──────
ax.text(
    ax.get_xlim()[0],
    log10_p - 0.3,
    "Negative change in drug\nresponse compared to control",
    ha="left", va="top", fontsize=9, color="#2980B9",
)

ax.text(
    ax.get_xlim()[1],
    log10_p - 0.3,
    "Positive change in drug\nresponse compared to control",
    ha="right", va="top", fontsize=9, color="#C0392B",
)
# ── Region annotation text ─────────────────────────────────────────────────────
region_x = df["delta_value"].min() - 0.1

ax.text(region_x, log10_p + 1.5,
        "Statistically significant\nchange in differential\ndrug response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

ax.text(region_x, 0.6,
        "Statistically insignificant\nchange in differential\ndrug response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color="#2980B9", label="Negative Drug Response"),
    mpatches.Patch(color="#C0392B", label="Positive Drug Response"),
    mpatches.Patch(color="#AAAAAA", label="Insignificant"),
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.01, 1),
          frameon=True, fontsize=9, edgecolor="#cccccc", borderaxespad=0)

# ── Axes labels & title ────────────────────────────────────────────────────────
ax.set_xlabel("Effect Size (delta_value)", fontsize=11)
ax.set_ylabel(r"$-\log_{10}$(P-value)", fontsize=11)

fig.suptitle("Volcano Plot: Drug Response (R175H vs WT)",
             x=0.02, ha="left", fontsize=16, fontweight="bold",
             y=0.98)

fig.add_artist(plt.Line2D(
    [0.01, 0.01], [0.91, 0.99],
    transform=fig.transFigure,
    color="black", linewidth=4,
))

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#bbbbbb")
ax.tick_params(colors="#555555")

plt.tight_layout(rect=[0, 0, 0.85, 0.93])
plt.savefig("volcano_plot_styled.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------- GENE EXPRESSION ----------------

# Load the gene response data
df = pd.read_csv("R175H_vs_WT_gene_results.csv")

# Calculate -log10(p-value)
df["neg_log10_p"] = -np.log10(df["p_value"])

# Set thresholds
p_threshold = 0.05
effect_threshold = 1

# Classify points
conditions = [
    (df["p_value"] < p_threshold) & (df["delta_value"] > effect_threshold),
    (df["p_value"] < p_threshold) & (df["delta_value"] < -effect_threshold)
]
choices = ["More Sensitive", "More Resistant"]
df["significance"] = np.select(conditions, choices, default="Not Significant")

# ── Figure & axes ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ── Scatter points ─────────────────────────────────────────────────────────────
color_map = {
    "More Sensitive": "#C0392B",
    "More Resistant": "#2980B9",
    "Not Significant": "#AAAAAA",
}

order = ["Not Significant", "More Resistant", "More Sensitive"]
for category in order:
    subset = df[df["significance"] == category]
    ax.scatter(
        subset["delta_value"],
        subset["neg_log10_p"],
        c=color_map[category],
        s=18,
        alpha=0.75,
        linewidths=0,
        zorder=3 if category != "Not Significant" else 2,
    )

# ── Threshold lines ────────────────────────────────────────────────────────────
log10_p = -np.log10(p_threshold)

ax.axhline(log10_p, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(-effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(0, color="#888888", linestyle="--", linewidth=0.9, zorder=1)

# ── X-axis range ───────────────────────────────────────────────────────────────
ax.set_xlim(-4, 3)

# ── Gene name labels (top 20 most significant only, within x range) ────────────
x_min, x_max = -4, 3
df_visible = df[(df["delta_value"] >= x_min) & (df["delta_value"] <= x_max)]
labeled = df_visible[df_visible["significance"] != "Not Significant"].nsmallest(20, "p_value")

texts = []
for _, row in labeled.iterrows():
    texts.append(ax.text(
        row["delta_value"],
        row["neg_log10_p"],
        row["feature"],
        fontsize=7,
        color=color_map[row["significance"]],
        va="bottom",
    ))

adjust_text(
    texts,
    x=df_visible["delta_value"].values,
    y=df_visible["neg_log10_p"].values,
    arrowprops=dict(
        arrowstyle="-",
        color="gray",
        lw=0.5,
        shrinkA=10,
        shrinkB=5,
    ),
    expand_points=(1.5, 1.5),
    expand_text=(1.2, 1.2),
    force_points=(0.5, 0.5),
    force_text=(0.5, 0.5),
)

# ── "Threshold" label ──────────────────────────────────────────────────────────
ax.text(
    3,
    log10_p + 0.05,
    "Threshold",
    color="#888888", fontsize=8, va="bottom", ha="right",
)

# ── Zero-point arrows ──────────────────────────────────────────────────────────
ax.annotate(
    "", xy=(0.55, 0.97), xytext=(0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.annotate(
    "", xy=(-0.55, 0.97), xytext=(-0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.text(0, 0.975, "Zero point", ha="center", va="bottom",
        transform=ax.get_xaxis_transform(), fontsize=8, color="gray")

# ── Top annotation labels ──────────────────────────────────────────────────────
ax.text(
    -4,
    log10_p - 0.3,
    "Negative change in gene\nresponse compared to control",
    ha="left", va="top", fontsize=9, color="#2980B9",
)

ax.text(
    3,
    log10_p - 0.3,
    "Positive change in gene\nresponse compared to control",
    ha="right", va="top", fontsize=9, color="#C0392B",
)

# ── Region annotation text ─────────────────────────────────────────────────────
ax.text(-3.9, log10_p + 1.5,
        "Statistically significant\nchange in differential\ngene response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

ax.text(-3.9, 0.6,
        "Statistically insignificant\nchange in differential\ngene response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color="#2980B9", label="Negative Gene Response"),
    mpatches.Patch(color="#C0392B", label="Positive Gene Response"),
    mpatches.Patch(color="#AAAAAA", label="Insignificant"),
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.01, 1),
          frameon=True, fontsize=9, edgecolor="#cccccc", borderaxespad=0)

# ── Axes labels & title ────────────────────────────────────────────────────────
ax.set_xlabel("Effect Size (delta_value)", fontsize=11)
ax.set_ylabel(r"$-\log_{10}$(P-value)", fontsize=11)

fig.suptitle("Volcano Plot: Gene Response (R175H vs WT)",
             x=0.02, ha="left", fontsize=16, fontweight="bold",
             y=0.98)

fig.add_artist(plt.Line2D(
    [0.01, 0.01], [0.91, 0.99],
    transform=fig.transFigure,
    color="black", linewidth=4,
))

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#bbbbbb")
ax.tick_params(colors="#555555")

plt.subplots_adjust(left=0.08, right=0.78, top=0.90, bottom=0.08)
plt.savefig("volcano_plot_gene_styled.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
''' Mutation: R248Q'''

# ---------------- DRUG RESPONSE ----------------

# Load the drug response data
df = pd.read_csv("R248Q_vs_WT_drug_results.csv")

# Calculate -log10(p-value)
df["neg_log10_p"] = -np.log10(df["p_value"])

# Set thresholds
p_threshold = 0.05
effect_threshold = 1

# Classify points
conditions = [
    (df["p_value"] < p_threshold) & (df["delta_value"] > effect_threshold),
    (df["p_value"] < p_threshold) & (df["delta_value"] < -effect_threshold)
]
choices = ["More Sensitive", "More Resistant"]
df["significance"] = np.select(conditions, choices, default="Not Significant")

# ── Figure & axes ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ── Scatter points ─────────────────────────────────────────────────────────────
color_map = {
    "More Sensitive": "#C0392B",
    "More Resistant": "#2980B9",
    "Not Significant": "#AAAAAA",
}

order = ["Not Significant", "More Resistant", "More Sensitive"]
for category in order:
    subset = df[df["significance"] == category]
    ax.scatter(
        subset["delta_value"],
        subset["neg_log10_p"],
        c=color_map[category],
        s=18,
        alpha=0.75,
        linewidths=0,
        zorder=3 if category != "Not Significant" else 2,
    )

# ── Threshold lines ────────────────────────────────────────────────────────────
log10_p = -np.log10(p_threshold)

ax.axhline(log10_p, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(-effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(0, color="#888888", linestyle="--", linewidth=0.9, zorder=1)

# ── Drug name labels ───────────────────────────────────────────────────────────
labeled = df[df["significance"] != "Not Significant"]

texts = []
for _, row in labeled.iterrows():
    texts.append(ax.text(
        row["delta_value"],
        row["neg_log10_p"],
        row["feature"],
        fontsize=7,
        color=color_map[row["significance"]],
        va="bottom",
    ))

adjust_text(
    texts,
    x=df["delta_value"].values,
    y=df["neg_log10_p"].values,
    arrowprops=dict(
        arrowstyle="-",
        color="gray",
        lw=0.5,
        shrinkA=5,
        shrinkB=5,
    ),
    expand_points=(1.5, 1.5),
    expand_text=(1.2, 1.2),
    force_points=(0.5, 0.5),
    force_text=(0.5, 0.5),
)

# ── "Threshold" label ──────────────────────────────────────────────────────────
ax.text(
    ax.get_xlim()[1] if ax.get_xlim()[1] > effect_threshold + 0.5 else effect_threshold + 0.5,
    log10_p + 0.05,
    "Threshold",
    color="#888888", fontsize=8, va="bottom", ha="right",
)

# ── Zero-point arrows ──────────────────────────────────────────────────────────
ax.annotate(
    "", xy=(0.55, 0.97), xytext=(0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.annotate(
    "", xy=(-0.55, 0.97), xytext=(-0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.text(0, 0.975, "Zero point", ha="center", va="bottom",
        transform=ax.get_xaxis_transform(), fontsize=8, color="gray")

# ── Top annotation labels (moved below threshold line, outside data area) ──────
ax.text(
    ax.get_xlim()[0],
    log10_p - 0.3,
    "Negative change in drug\nresponse compared to control",
    ha="left", va="top", fontsize=9, color="#2980B9",
)

ax.text(
    ax.get_xlim()[1],
    log10_p - 0.3,
    "Positive change in drug\nresponse compared to control",
    ha="right", va="top", fontsize=9, color="#C0392B",
)

# ── Region annotation text ─────────────────────────────────────────────────────
region_x = df["delta_value"].min() - 0.1

ax.text(region_x, log10_p + 1.5,
        "Statistically significant\nchange in differential\ndrug response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

ax.text(region_x, 0.6,
        "Statistically insignificant\nchange in differential\ndrug response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color="#2980B9", label="Negative Drug Response"),
    mpatches.Patch(color="#C0392B", label="Positive Drug Response"),
    mpatches.Patch(color="#AAAAAA", label="Insignificant"),
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.01, 1),
          frameon=True, fontsize=9, edgecolor="#cccccc", borderaxespad=0)

# ── Axes labels & title ────────────────────────────────────────────────────────
ax.set_xlabel("Effect Size (delta_value)", fontsize=11)
ax.set_ylabel(r"$-\log_{10}$(P-value)", fontsize=11)

fig.suptitle("Volcano Plot: Drug Response (R248Q vs WT)",
             x=0.02, ha="left", fontsize=16, fontweight="bold",
             y=0.98)

fig.add_artist(plt.Line2D(
    [0.01, 0.01], [0.91, 0.99],
    transform=fig.transFigure,
    color="black", linewidth=4,
))

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#bbbbbb")
ax.tick_params(colors="#555555")

plt.tight_layout(rect=[0, 0, 0.85, 0.93])
plt.savefig("volcano_plot_styled.png", dpi=150, bbox_inches="tight")
plt.show()


# ---------------- GENE EXPRESSION ----------------

# Load the gene response data
df = pd.read_csv("R248Q_vs_WT_gene_results.csv")

# Calculate -log10(p-value)
df["neg_log10_p"] = -np.log10(df["p_value"])

# Set thresholds
p_threshold = 0.05
effect_threshold = 1

# Classify points
conditions = [
    (df["p_value"] < p_threshold) & (df["delta_value"] > effect_threshold),
    (df["p_value"] < p_threshold) & (df["delta_value"] < -effect_threshold)
]
choices = ["More Sensitive", "More Resistant"]
df["significance"] = np.select(conditions, choices, default="Not Significant")

# ── Figure & axes ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ── Scatter points ─────────────────────────────────────────────────────────────
color_map = {
    "More Sensitive": "#C0392B",
    "More Resistant": "#2980B9",
    "Not Significant": "#AAAAAA",
}

order = ["Not Significant", "More Resistant", "More Sensitive"]
for category in order:
    subset = df[df["significance"] == category]
    ax.scatter(
        subset["delta_value"],
        subset["neg_log10_p"],
        c=color_map[category],
        s=18,
        alpha=0.75,
        linewidths=0,
        zorder=3 if category != "Not Significant" else 2,
    )

# ── Threshold lines ────────────────────────────────────────────────────────────
log10_p = -np.log10(p_threshold)

ax.axhline(log10_p, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(-effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(0, color="#888888", linestyle="--", linewidth=0.9, zorder=1)

# ── X-axis range ───────────────────────────────────────────────────────────────
ax.set_xlim(-4, 3)

# ── Gene name labels (top 20 most significant only, within x range) ────────────
x_min, x_max = -4, 3
df_visible = df[(df["delta_value"] >= x_min) & (df["delta_value"] <= x_max)]
labeled = df_visible[df_visible["significance"] != "Not Significant"].nsmallest(20, "p_value")

texts = []
for _, row in labeled.iterrows():
    texts.append(ax.text(
        row["delta_value"],
        row["neg_log10_p"],
        row["feature"],
        fontsize=7,
        color=color_map[row["significance"]],
        va="bottom",
    ))

adjust_text(
    texts,
    x=df_visible["delta_value"].values,
    y=df_visible["neg_log10_p"].values,
    arrowprops=dict(
        arrowstyle="-",
        color="gray",
        lw=0.5,
        shrinkA=10,
        shrinkB=5,
    ),
    expand_points=(1.5, 1.5),
    expand_text=(1.2, 1.2),
    force_points=(0.5, 0.5),
    force_text=(0.5, 0.5),
)

# ── "Threshold" label ──────────────────────────────────────────────────────────
ax.text(
    3,
    log10_p + 0.05,
    "Threshold",
    color="#888888", fontsize=8, va="bottom", ha="right",
)

# ── Zero-point arrows ──────────────────────────────────────────────────────────
ax.annotate(
    "", xy=(0.55, 0.97), xytext=(0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.annotate(
    "", xy=(-0.55, 0.97), xytext=(-0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.text(0, 0.975, "Zero point", ha="center", va="bottom",
        transform=ax.get_xaxis_transform(), fontsize=8, color="gray")

# ── Top annotation labels ──────────────────────────────────────────────────────
ax.text(
    -4,
    log10_p - 0.3,
    "Negative change in gene\nresponse compared to control",
    ha="left", va="top", fontsize=9, color="#2980B9",
)

ax.text(
    3,
    log10_p - 0.3,
    "Positive change in gene\nresponse compared to control",
    ha="right", va="top", fontsize=9, color="#C0392B",
)

# ── Region annotation text ─────────────────────────────────────────────────────
ax.text(-3.9, log10_p + 1.5,
        "Statistically significant\nchange in differential\ngene response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

ax.text(-3.9, 0.6,
        "Statistically insignificant\nchange in differential\ngene response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color="#2980B9", label="Negative Gene Response"),
    mpatches.Patch(color="#C0392B", label="Positive Gene Response"),
    mpatches.Patch(color="#AAAAAA", label="Insignificant"),
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.01, 1),
          frameon=True, fontsize=9, edgecolor="#cccccc", borderaxespad=0)

# ── Axes labels & title ────────────────────────────────────────────────────────
ax.set_xlabel("Effect Size (delta_value)", fontsize=11)
ax.set_ylabel(r"$-\log_{10}$(P-value)", fontsize=11)

fig.suptitle("Volcano Plot: Gene Response (R248Q vs WT)",
             x=0.02, ha="left", fontsize=16, fontweight="bold",
             y=0.98)

fig.add_artist(plt.Line2D(
    [0.01, 0.01], [0.91, 0.99],
    transform=fig.transFigure,
    color="black", linewidth=4,
))

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#bbbbbb")
ax.tick_params(colors="#555555")

plt.subplots_adjust(left=0.08, right=0.78, top=0.90, bottom=0.08)
plt.savefig("volcano_plot_gene_styled.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
''' Mutation: R273H'''

# ---------------- DRUG RESPONSE ----------------

# Load the drug response data
df = pd.read_csv("R273H_vs_WT_drug_results.csv")

# Calculate -log10(p-value)
df["neg_log10_p"] = -np.log10(df["p_value"])

# Set thresholds
p_threshold = 0.05
effect_threshold = 1

# Classify points
conditions = [
    (df["p_value"] < p_threshold) & (df["delta_value"] > effect_threshold),
    (df["p_value"] < p_threshold) & (df["delta_value"] < -effect_threshold)
]
choices = ["More Sensitive", "More Resistant"]
df["significance"] = np.select(conditions, choices, default="Not Significant")

# ── Figure & axes ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ── Scatter points ─────────────────────────────────────────────────────────────
color_map = {
    "More Sensitive": "#C0392B",
    "More Resistant": "#2980B9",
    "Not Significant": "#AAAAAA",
}

order = ["Not Significant", "More Resistant", "More Sensitive"]
for category in order:
    subset = df[df["significance"] == category]
    ax.scatter(
        subset["delta_value"],
        subset["neg_log10_p"],
        c=color_map[category],
        s=18,
        alpha=0.75,
        linewidths=0,
        zorder=3 if category != "Not Significant" else 2,
    )

# ── Threshold lines ────────────────────────────────────────────────────────────
log10_p = -np.log10(p_threshold)

ax.axhline(log10_p, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(-effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(0, color="#888888", linestyle="--", linewidth=0.9, zorder=1)

# ── Drug name labels ───────────────────────────────────────────────────────────
labeled = df[df["significance"] != "Not Significant"]

texts = []
for _, row in labeled.iterrows():
    texts.append(ax.text(
        row["delta_value"],
        row["neg_log10_p"],
        row["feature"],
        fontsize=7,
        color=color_map[row["significance"]],
        va="bottom",
    ))

adjust_text(
    texts,
    x=df["delta_value"].values,
    y=df["neg_log10_p"].values,
    arrowprops=dict(
        arrowstyle="-",
        color="gray",
        lw=0.5,
        shrinkA=5,
        shrinkB=5,
    ),
    expand_points=(1.5, 1.5),
    expand_text=(1.2, 1.2),
    force_points=(0.5, 0.5),
    force_text=(0.5, 0.5),
)

# ── "Threshold" label ──────────────────────────────────────────────────────────
ax.text(
    ax.get_xlim()[1] if ax.get_xlim()[1] > effect_threshold + 0.5 else effect_threshold + 0.5,
    log10_p + 0.05,
    "Threshold",
    color="#888888", fontsize=8, va="bottom", ha="right",
)

# ── Zero-point arrows ──────────────────────────────────────────────────────────
ax.annotate(
    "", xy=(0.55, 0.97), xytext=(0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.annotate(
    "", xy=(-0.55, 0.97), xytext=(-0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.text(0, 0.975, "Zero point", ha="center", va="bottom",
        transform=ax.get_xaxis_transform(), fontsize=8, color="gray")

# ── Top annotation labels ──────────────────────────────────────────────────────
y_max = df["neg_log10_p"].max()

ax.text(
    -effect_threshold - 0.1, y_max * 0.75,
    "Negative change in drug\nresponse compared to control",
    ha="right", va="top", fontsize=9, color="#2980B9",
)

ax.text(
    effect_threshold + 0.1, y_max * 0.75,
    "Positive change in drug\nresponse compared to control",
    ha="left", va="top", fontsize=9, color="#C0392B",
)

# ── Region annotation text ─────────────────────────────────────────────────────
region_x = df["delta_value"].min() - 0.1

ax.text(region_x, log10_p + 1.5,
        "Statistically significant\nchange in differential\ndrug response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

ax.text(region_x, 0.6,
        "Statistically insignificant\nchange in differential\ndrug response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color="#2980B9", label="Negative Drug Response"),
    mpatches.Patch(color="#C0392B", label="Positive Drug Response"),
    mpatches.Patch(color="#AAAAAA", label="Insignificant"),
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.01, 1),
          frameon=True, fontsize=9, edgecolor="#cccccc", borderaxespad=0)

# ── Axes labels & title ────────────────────────────────────────────────────────
ax.set_xlabel("Effect Size (delta_value)", fontsize=11)
ax.set_ylabel(r"$-\log_{10}$(P-value)", fontsize=11)

fig.suptitle("Volcano Plot: Drug Response (R273H vs WT)",
             x=0.02, ha="left", fontsize=16, fontweight="bold",
             y=0.98)

fig.add_artist(plt.Line2D(
    [0.01, 0.01], [0.91, 0.99],
    transform=fig.transFigure,
    color="black", linewidth=4,
))

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#bbbbbb")
ax.tick_params(colors="#555555")

plt.tight_layout(rect=[0, 0, 0.85, 0.93])
plt.savefig("volcano_plot_styled.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------------- GENE EXPRESSION ----------------

# Load the gene response data
df = pd.read_csv("R273H_vs_WT_gene_results.csv")

# Calculate -log10(p-value)
df["neg_log10_p"] = -np.log10(df["p_value"])

# Set thresholds
p_threshold = 0.05
effect_threshold = 1

# Classify points
conditions = [
    (df["p_value"] < p_threshold) & (df["delta_value"] > effect_threshold),
    (df["p_value"] < p_threshold) & (df["delta_value"] < -effect_threshold)
]
choices = ["More Sensitive", "More Resistant"]
df["significance"] = np.select(conditions, choices, default="Not Significant")

# ── Figure & axes ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ── Scatter points ─────────────────────────────────────────────────────────────
color_map = {
    "More Sensitive": "#C0392B",
    "More Resistant": "#2980B9",
    "Not Significant": "#AAAAAA",
}

order = ["Not Significant", "More Resistant", "More Sensitive"]
for category in order:
    subset = df[df["significance"] == category]
    ax.scatter(
        subset["delta_value"],
        subset["neg_log10_p"],
        c=color_map[category],
        s=18,
        alpha=0.75,
        linewidths=0,
        zorder=3 if category != "Not Significant" else 2,
    )

# ── Threshold lines ────────────────────────────────────────────────────────────
log10_p = -np.log10(p_threshold)

ax.axhline(log10_p, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(-effect_threshold, color="#888888", linestyle="--", linewidth=0.9, zorder=1)
ax.axvline(0, color="#888888", linestyle="--", linewidth=0.9, zorder=1)

# ── X-axis range ───────────────────────────────────────────────────────────────
ax.set_xlim(-4, 3)

# ── Gene name labels (top 20 most significant only, within x range) ────────────
x_min, x_max = -4, 3
df_visible = df[(df["delta_value"] >= x_min) & (df["delta_value"] <= x_max)]
labeled = df_visible[df_visible["significance"] != "Not Significant"].nsmallest(20, "p_value")

texts = []
for _, row in labeled.iterrows():
    texts.append(ax.text(
        row["delta_value"],
        row["neg_log10_p"],
        row["feature"],
        fontsize=7,
        color=color_map[row["significance"]],
        va="bottom",
    ))

adjust_text(
    texts,
    x=df_visible["delta_value"].values,
    y=df_visible["neg_log10_p"].values,
    arrowprops=dict(
        arrowstyle="-",
        color="gray",
        lw=0.5,
        shrinkA=10,
        shrinkB=5,
    ),
    expand_points=(1.5, 1.5),
    expand_text=(1.2, 1.2),
    force_points=(0.5, 0.5),
    force_text=(0.5, 0.5),
)

# ── "Threshold" label ──────────────────────────────────────────────────────────
ax.text(
    3,
    log10_p + 0.05,
    "Threshold",
    color="#888888", fontsize=8, va="bottom", ha="right",
)

# ── Zero-point arrows ──────────────────────────────────────────────────────────
ax.annotate(
    "", xy=(0.55, 0.97), xytext=(0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.annotate(
    "", xy=(-0.55, 0.97), xytext=(-0.05, 0.97),
    xycoords=("data", "axes fraction"), textcoords=("data", "axes fraction"),
    arrowprops=dict(arrowstyle="->", color="gray", lw=1),
)
ax.text(0, 0.975, "Zero point", ha="center", va="bottom",
        transform=ax.get_xaxis_transform(), fontsize=8, color="gray")

# ── Top annotation labels ──────────────────────────────────────────────────────
ax.text(
    -4,
    log10_p - 0.3,
    "Negative change in gene\nresponse compared to control",
    ha="left", va="top", fontsize=9, color="#2980B9",
)

ax.text(
    3,
    log10_p - 0.3,
    "Positive change in gene\nresponse compared to control",
    ha="right", va="top", fontsize=9, color="#C0392B",
)

# ── Region annotation text ─────────────────────────────────────────────────────
ax.text(-3.9, log10_p + 1.5,
        "Statistically significant\nchange in differential\ngene response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

ax.text(-3.9, 0.6,
        "Statistically insignificant\nchange in differential\ngene response",
        fontsize=8, color="black", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"))

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color="#2980B9", label="Negative Gene Response"),
    mpatches.Patch(color="#C0392B", label="Positive Gene Response"),
    mpatches.Patch(color="#AAAAAA", label="Insignificant"),
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.01, 1),
          frameon=True, fontsize=9, edgecolor="#cccccc", borderaxespad=0)

# ── Axes labels & title ────────────────────────────────────────────────────────
ax.set_xlabel("Effect Size (delta_value)", fontsize=11)
ax.set_ylabel(r"$-\log_{10}$(P-value)", fontsize=11)

fig.suptitle("Volcano Plot: Gene Response (R273H vs WT)",
             x=0.02, ha="left", fontsize=16, fontweight="bold",
             y=0.98)

fig.add_artist(plt.Line2D(
    [0.01, 0.01], [0.91, 0.99],
    transform=fig.transFigure,
    color="black", linewidth=4,
))

ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#bbbbbb")
ax.tick_params(colors="#555555")

plt.subplots_adjust(left=0.08, right=0.78, top=0.90, bottom=0.08)
plt.savefig("volcano_plot_gene_styled.png", dpi=150, bbox_inches="tight")
plt.show()


## Violin Plots

In [ ]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# ----------------------------
# 1. File paths
# ----------------------------
files = {
    "WT": "wildtype_drug_response.tsv",
    "R175H": "R175H_drug_response.tsv",
    "R248Q": "R248Q_drug_response.tsv",
    "R273H": "R273H_drug_response.tsv"
}

# ----------------------------
# 2. Function to reshape one file
# ----------------------------
def load_and_melt(file_path, mutation_label):
    """
    Reads a drug-response matrix TSV file and converts it to long format.

    Expected input format:
    - first column = drug names
    - remaining columns = cell lines
    - values = drug response scores
    """
    df = pd.read_csv(file_path, sep="\t")

    # Rename first column to Drug if needed
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Drug"})

    # Convert wide format -> long format
    long_df = df.melt(
        id_vars="Drug",
        var_name="CellLine",
        value_name="Response"
    )

    # Add mutation label
    long_df["Mutation"] = mutation_label

    return long_df

# ----------------------------
# 3. Load all files and combine
# ----------------------------
all_data = []

for mutation, file_path in files.items():
    temp = load_and_melt(file_path, mutation)
    all_data.append(temp)

combined_df = pd.concat(all_data, ignore_index=True)

# Optional: remove missing values
combined_df = combined_df.dropna(subset=["Response"])

# Optional: make mutation order consistent
mutation_order = ["WT", "R175H", "R248Q", "R273H"]
combined_df["Mutation"] = pd.Categorical(
    combined_df["Mutation"],
    categories=mutation_order,
    ordered=True
)

# Save combined data if you want
combined_df.to_csv("combined_drug_response_long.tsv", sep="\t", index=False)

print("Combined dataframe shape:", combined_df.shape)
print(combined_df.head())

# ----------------------------
# 4. Function to make one violin plot for one drug
# ----------------------------
def plot_one_drug(df, drug_name, output_folder="violin_plots"):
    """
    Makes a violin plot for one drug across mutation groups
    and adds p-values comparing WT to each mutant.
    """
    os.makedirs(output_folder, exist_ok=True)

    drug_df = df[df["Drug"] == drug_name].copy()

    if drug_df.empty:
        print(f"No data found for drug: {drug_name}")
        return

    mutation_order = ["WT", "R175H", "R248Q", "R273H"]

    palette = {
        "WT": "#4C72B0",
        "R175H": "#DD8452",
        "R248Q": "#55A868",
        "R273H": "#C44E52"
    }

    plt.figure(figsize=(9, 7))

    # Violin plot
    sns.violinplot(
        data=drug_df,
        x="Mutation",
        y="Response",
        order=mutation_order,
        palette=palette,
        inner="box",
        cut=0
    )

    # Add individual data points
    sns.stripplot(
        data=drug_df,
        x="Mutation",
        y="Response",
        order=mutation_order,
        color="black",
        size=3,
        alpha=0.5,
        jitter=True
    )

    # ----------------------------
    # Compute p-values: WT vs each mutant
    # ----------------------------
    wt_vals = drug_df.loc[drug_df["Mutation"] == "WT", "Response"].dropna()

    comparisons = ["R175H", "R248Q", "R273H"]
    pval_labels = {}

    for mutant in comparisons:
        mutant_vals = drug_df.loc[drug_df["Mutation"] == mutant, "Response"].dropna()

        if len(wt_vals) > 1 and len(mutant_vals) > 1:
            tstat, pval = ttest_ind(wt_vals, mutant_vals, equal_var=False)

            if pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "ns"

            pval_labels[mutant] = f"p={pval:.3g}\n{sig}"
        else:
            pval_labels[mutant] = "not enough\ndata"

    # ----------------------------
    # Put p-values below mutant plots
    # ----------------------------
    ymin = drug_df["Response"].min()
    ymax = drug_df["Response"].max()
    yrange = ymax - ymin

    # Add extra space below plot
    bottom_text_y = ymin - 0.18 * yrange
    plt.ylim(ymin - 0.28 * yrange, ymax + 0.08 * yrange)

    # x positions: WT=0, R175H=1, R248Q=2, R273H=3
    x_positions = {
        "R175H": 1,
        "R248Q": 2,
        "R273H": 3
    }

    for mutant, xpos in x_positions.items():
        plt.text(
            xpos,
            bottom_text_y,
            pval_labels[mutant],
            ha="center",
            va="top",
            fontsize=10
        )

    plt.title(f"{drug_name} response across TP53 mutation groups")
    plt.xlabel("TP53 Mutation Group")
    plt.ylabel("Drug Response")
    plt.tight_layout()

    safe_name = drug_name.replace("/", "_").replace(" ", "_")
    out_path = os.path.join(output_folder, f"{safe_name}_violin.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved: {out_path}")


# ----------------------------
# 5. Make plots for selected drugs
# ----------------------------
# Replace these with the drugs you actually want to plot
drugs_to_plot = [
    "AMG-232", # DRUG RESPONSE R175H VS WT
    "MELPHALAN"
    "MILADEMETAN",
    "IDASANUTLIN"
]

for drug in drugs_to_plot:
    plot_one_drug(combined_df, drug)

# ----------------------------
# 6. Optional: automatically plot top N drugs
# ----------------------------
# If you just want to test the code on the first 5 drugs:
top_test_drugs = combined_df["Drug"].drop_duplicates().head(5)

for drug in top_test_drugs:
    plot_one_drug(combined_df, drug, output_folder="test_violin_plots")